<a href="https://colab.research.google.com/github/guneetnagia/LearnML/blob/main/keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import keras
from keras.layers import Input, Dense
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [16]:
iris = load_iris()
X,y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_test = scaler.fit_transform(X_test)

In [18]:
model = keras.Sequential()
model.add(Input(shape=(4,)))
model.add(Dense(8,activation='relu'))
model.add(Dense(10,activation='relu'))
model.add(Dense(10, activation='relu'))
model.add(Dense(3, activation='softmax'))

In [19]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [20]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │            90 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 273 (1.07 KB)

 Trainable params: 273 (1.07 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=16, verbose=1)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.3958 - loss: 1.0920 - val_accuracy: 0.4167 - val_loss: 1.0902
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4479 - loss: 1.0813 - val_accuracy: 0.4583 - val_loss: 1.0745
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4896 - loss: 1.0713 - val_accuracy: 0.4583 - val_loss: 1.0587
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5312 - loss: 1.0605 - val_accuracy: 0.4583 - val_loss: 1.0448
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5521 - loss: 1.0507 - val_accuracy: 0.4583 - val_loss: 1.0284
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5625 - loss: 1.0383 - val_accuracy: 0.4583 - val_loss: 1.0125
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5625 - loss: 1.0261 - val_accuracy: 0.4583 - val_loss: 0.9944
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5729 - loss: 1.0133 - val_accuracy: 0.5000 - val_loss: 0.9753


In [22]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss:{loss}, Accuracy:{accuracy}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.3667 - loss: 2.4215
Loss:2.421499490737915, Accuracy:0.36666667461395264


In [23]:
model.save('iris_mlp_guneet.keras')

In [28]:
reloaded = keras.models.load_model('iris_mlp_guneet.keras')
sample = scaler.transform([[5.1, 3.5, 1.4, 0.2]])
probs = reloaded.predict(sample)[0]
print(f"sample:{sample}")
print(f'Prediction: {iris.target_names[np.argmax(probs)]}(reloaded.predict(sample))')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
sample:[[-1.05919085  1.21615408 -1.37173756 -1.33370109]]
Prediction: setosa(reloaded.predict(sample))


In [30]:
import gradio as gr
import numpy as np
CLASSES = list(iris.target_names)
def predict(sepal_length, sepal_width, petal_length, petal_width):
  X = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
  probs = model.predict(scaler.transform(X))[0]
  return {CLASSES[i]:float(probs[i]) for i in range(3)}

In [32]:
from flax.nnx.nn.recurrent import Output
demo=gr.Interface(fn=predict, inputs= [gr.Number(), gr.Number(),gr.Number(),gr.Number()],
                  outputs=gr.Label(num_top_classes=3, label="predicted species"))

In [33]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b8b57d3af83a2f756.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
